In [1]:
import os
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns

from IPython.display import display

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data.cleaner import DataCleaner
from src.features.builder import build_rfm_features, compute_product_return_rate
from src.mining.association import mine_return_association_rules

sns.set(style="whitegrid")

# 1. Đảm bảo có dữ liệu sạch
cleaner = DataCleaner(config_path="../configs/params.yaml")
raw_df, clean_df = cleaner.run_full_cleaning(save=True)

# 2. Xây dựng đặc trưng RFM và ReturnRate
rfm_df = build_rfm_features(clean_df, config_path="../configs/params.yaml")
product_return_df = compute_product_return_rate(clean_df, config_path="../configs/params.yaml")

print("=== Một vài dòng đầu tiên của bảng RFM (Customer) ===")
display(rfm_df.head())

print("\n=== Một vài dòng đầu tiên của bảng ReturnRate theo sản phẩm ===")
display(product_return_df.head())


2026-03-17 15:22:51,468 - src.data.cleaner - INFO - Đã load cấu hình từ ..\configs\params.yaml


2026-03-17 15:22:51,471 - src.data.cleaner - INFO - Đang đọc dữ liệu từ E:\DLL\btl\data\raw\data.csv


2026-03-17 15:22:52,581 - src.data.cleaner - INFO - Đã load 541909 dòng, 8 cột


2026-03-17 15:22:52,595 - src.data.cleaner - INFO - Đang loại bỏ 135080 dòng thiếu CustomerID theo cấu hình.


2026-03-17 15:22:52,783 - src.data.cleaner - INFO - Hoàn tất xử lý missing. Còn lại 406829 dòng.


...

2026-03-17 15:22:57,163 - src.features.builder - INFO - Đã tạo bảng product_return_rate cho 3684 sản phẩm.


=== Một vài dòng đầu tiên của bảng RFM (Customer) ===


,CustomerID,frequency,monetary,recency,return_rate_customer
0,12346.0,2,249.60,325,0.5
1,12347.0,7,4185.20,129,0.0
2,12348.0,4,1530.48,75,0.0
3,12349.0,1,1443.80,18,0.0
4,12350.0,1,309.40,310,0.0



=== Một vài dòng đầu tiên của bảng ReturnRate theo sản phẩm ===


,StockCode,product_return_rate
0,10002,0.0
1,10080,0.0
2,10120,0.0
3,10123C,0.0
4,10124A,0.0


In [2]:
# 3. Phân cụm khách hàng (KMeans) + Profiling cụm
# Mục tiêu: phân nhóm khách hàng theo hành vi mua hàng và tỷ lệ trả hàng

import yaml
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Đọc số cụm từ cấu hình
with open("../configs/params.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

n_clusters = int(cfg.get("clustering", {}).get("n_clusters", 4))

# Chọn feature cho clustering
cluster_features = rfm_df[["recency", "frequency", "monetary", "return_rate_customer"]].copy()
cluster_features = cluster_features.fillna(0)

scaler = StandardScaler()
X = scaler.fit_transform(cluster_features)

kmeans = KMeans(n_clusters=n_clusters, random_state=int(cfg.get("random_seed", 42)), n_init=10)
rfm_df["cluster"] = kmeans.fit_predict(X)

# Profiling: thống kê trung bình theo cụm
cluster_profile = (
    rfm_df.groupby("cluster")
    .agg(
        n_customers=("CustomerID", "count"),
        recency_mean=("recency", "mean"),
        frequency_mean=("frequency", "mean"),
        monetary_mean=("monetary", "mean"),
        return_rate_mean=("return_rate_customer", "mean"),
    )
    .sort_values("return_rate_mean", ascending=False)
)

print("=== Profiling cụm (trung bình theo cụm) ===")
display(cluster_profile)


=== Profiling cụm (trung bình theo cụm) ===


,n_customers,recency_mean,frequency_mean,monetary_mean,return_rate_mean
cluster,,,,,
0,67,234.432836,2.059701,427.791194,0.760783
3,20,85.600000,91.200000,72295.361500,0.045038
1,1317,254.921792,2.337130,624.269377,0.020998
2,2967,85.005056,5.778901,1870.481678,0.020423


### Nhận xét & đặt tên cụm khách hàng (Customer Clusters)

Dựa trên bảng profiling (giá trị trung bình theo cụm), ta có thể đặt tên/mô tả các nhóm như sau (tham khảo, có thể điều chỉnh theo số cụm thực tế):

- **Cụm có `return_rate_mean` cao nhất**: *Nhóm “Return-prone” (chuyên trả hàng)*
  - Đặc điểm: tỷ lệ trả hàng cao bất thường; thường cần can thiệp chính sách (xác nhận size/kỳ vọng), kiểm tra chất lượng sản phẩm hoặc kiểm soát gian lận.
  - Hành động gợi ý: nhắc khách chọn đúng thông tin sản phẩm, cải thiện mô tả/hình ảnh, áp dụng rule kiểm tra thêm trước khi hoàn tiền.

- **Cụm có `monetary_mean` và `frequency_mean` cao nhưng `return_rate_mean` thấp**: *Nhóm “VIP/High-value”*
  - Đặc điểm: mua nhiều, mua thường xuyên và ít trả.
  - Hành động gợi ý: ưu tiên chăm sóc (ưu đãi, loyalty), cross-sell/upsell vì rủi ro trả thấp.

- **Cụm có `recency_mean` cao (lâu không mua)**: *Nhóm “Inactive/Churn risk”*
  - Đặc điểm: khách lâu không quay lại, giá trị có thể thấp hơn.
  - Hành động gợi ý: chiến dịch tái kích hoạt, email/khuyến mại theo sở thích, khảo sát lý do rời bỏ.

- **Cụm còn lại**: *Nhóm “Regular/Standard”*
  - Đặc điểm: hành vi ở mức trung bình.
  - Hành động gợi ý: tối ưu trải nghiệm mua hàng, gợi ý sản phẩm phù hợp để tăng Frequency/Monetary.

> Lưu ý: cách đặt tên cụm dựa trên tương quan giữa `recency`, `frequency`, `monetary`, và `return_rate_customer`. Khi báo cáo, bạn nên trích 1–2 con số tiêu biểu từ bảng profiling để làm luận cứ cho từng mô tả.


In [3]:
# Tự động gợi ý tên cụm dựa trên profiling (dựa trên percentile)
from IPython.display import Markdown

cp = cluster_profile.copy()

# Ngưỡng dựa trên phân vị để gán nhãn tương đối trong tập dữ liệu
ret_hi = cp["return_rate_mean"].quantile(0.75)
mon_hi = cp["monetary_mean"].quantile(0.75)
freq_hi = cp["frequency_mean"].quantile(0.75)
rec_hi = cp["recency_mean"].quantile(0.75)

cluster_names = {}
for cl, row in cp.iterrows():
    if row["return_rate_mean"] >= ret_hi:
        name = "Return-prone"
    elif (row["monetary_mean"] >= mon_hi) and (row["frequency_mean"] >= freq_hi) and (row["return_rate_mean"] < ret_hi):
        name = "VIP/High-value"
    elif row["recency_mean"] >= rec_hi:
        name = "Inactive/Churn risk"
    else:
        name = "Regular/Standard"
    cluster_names[int(cl)] = name

md_lines = ["### Tên cụm gợi ý (dựa trên số liệu profiling)"]
for cl, name in cluster_names.items():
    r = cp.loc[cl]
    md_lines.append(
        f"- **Cluster {cl} – {name}**: n={int(r['n_customers'])}, recency≈{r['recency_mean']:.1f}, freq≈{r['frequency_mean']:.1f}, monetary≈{r['monetary_mean']:.1f}, return_rate≈{r['return_rate_mean']:.3f}"
    )

display(Markdown("\n".join(md_lines)))


### Tên cụm gợi ý (dựa trên số liệu profiling)
- **Cluster 0 – Return-prone**: n=67, recency≈234.4, freq≈2.1, monetary≈427.8, return_rate≈0.761
- **Cluster 3 – VIP/High-value**: n=20, recency≈85.6, freq≈91.2, monetary≈72295.4, return_rate≈0.045
- **Cluster 1 – Inactive/Churn risk**: n=1317, recency≈254.9, freq≈2.3, monetary≈624.3, return_rate≈0.021
- **Cluster 2 – Regular/Standard**: n=2967, recency≈85.0, freq≈5.8, monetary≈1870.5, return_rate≈0.020

In [4]:
# 4. Luật kết hợp hướng tới nhãn trả hàng (consequent = is_return=1)
# Ý tưởng: đưa item `is_return=1` vào basket theo hóa đơn và khai phá các luật dự đoán trả hàng.

from src.mining.association import mine_rules_consequent_is_return

# Vì `is_return=1` khá hiếm, ta giảm min_support và giới hạn độ dài itemset để tránh quá nặng.
rules_return = mine_rules_consequent_is_return(
    clean_df,
    config_path="../configs/params.yaml",
    top_k=10,
    target_item="is_return=1",
    min_support=0.001,
    min_confidence=0.05,
    min_lift=1.0,
    max_len=2,
    max_items=200,
)

print("=== Top 10 luật có consequents chứa is_return=1 (Lift cao nhất) ===")
display(rules_return)


2026-03-17 15:23:01,516 - src.mining.association - INFO - Đã load cấu hình từ ..\configs\params.yaml


2026-03-17 15:23:04,175 - src.mining.association - INFO - Basket (invoice) có 22186 hóa đơn và 3886 items (gồm target).


2026-03-17 15:23:04,563 - src.mining.association - INFO - Đã giới hạn basket còn 200 sản phẩm phổ biến + target.


=== Top 10 luật có consequents chứa is_return=1 (Lift cao nhất) ===


,antecedents,consequents,support,confidence,lift
20807,Manual,is_return=1,0.006941,0.378378,2.297401


### Giải thích luật kết hợp (lọc consequent = `is_return=1`)

Ở phần trên, ta đã khai phá các luật dạng:

- **Antecedents (điều kiện)**: tập sản phẩm (hoặc combo sản phẩm) trong một hóa đơn.
- **Consequents (hậu quả)**: chứa item `is_return=1` nghĩa là hóa đơn có trả hàng.

Do đó, luật có Lift cao có thể được hiểu là:

- Khi hóa đơn chứa các sản phẩm ở vế **Antecedents**, thì **khả năng hóa đơn thuộc nhóm trả hàng** tăng lên đáng kể so với xác suất trả hàng trung bình.

Bên dưới, ta sẽ tự động trích **Top 3 luật có Lift cao nhất** và diễn giải ý nghĩa.

In [5]:
# Diễn giải Top 3 luật Lift cao nhất (consequent = is_return=1)
from IPython.display import Markdown

if rules_return is None or rules_return.empty:
    display(Markdown("**Không tìm thấy luật nào thỏa điều kiện consequent = `is_return=1`.**\n\nGợi ý: giảm `min_support` hoặc `min_confidence` trong `configs/params.yaml` để sinh thêm luật."))
else:
    k = min(3, len(rules_return))
    topk = rules_return.sort_values("lift", ascending=False).head(k).copy()
    display(topk)

    lines = [f"### Top {k} luật Lift cao nhất & ý nghĩa"]
    for i, row in enumerate(topk.itertuples(index=False), start=1):
        antecedents = row.antecedents
        support = float(row.support)
        conf = float(row.confidence)
        lift = float(row.lift)
        lines.append(
            f"**Luật {i}**: Nếu hóa đơn chứa **{{{antecedents}}}** thì có xu hướng thuộc nhóm **trả hàng** (`is_return=1`).\\n"
            f"- Support: **{support:.4f}**\\n"
            f"- Confidence: **{conf:.4f}** (xác suất trả hàng có điều kiện)\\n"
            f"- Lift: **{lift:.3f}** (mức tăng so với xác suất trả hàng trung bình)\\n"
            f"**Diễn giải**: combo sản phẩm này là một **tín hiệu rủi ro**; cần kiểm tra mô tả/hình ảnh/đóng gói/chất lượng hoặc quy trình tư vấn để giảm trả hàng."
        )

    display(Markdown("\n\n".join(lines)))


,antecedents,consequents,support,confidence,lift
20807,Manual,is_return=1,0.006941,0.378378,2.297401


### Top 1 luật Lift cao nhất & ý nghĩa

**Luật 1**: Nếu hóa đơn chứa **{Manual}** thì có xu hướng thuộc nhóm **trả hàng** (`is_return=1`).\n- Support: **0.0069**\n- Confidence: **0.3784** (xác suất trả hàng có điều kiện)\n- Lift: **2.297** (mức tăng so với xác suất trả hàng trung bình)\n**Diễn giải**: combo sản phẩm này là một **tín hiệu rủi ro**; cần kiểm tra mô tả/hình ảnh/đóng gói/chất lượng hoặc quy trình tư vấn để giảm trả hàng.